# AIRPATH-AI Milestone 3A — spatial PM2.5 estimation

This executable notebook reproduces the development-only spatial analysis implemented in `src/spatial_estimation.py`.

Protocol:

- coordinates come from HealthyAir/Data in Brief Table 1 (doi:10.1016/j.dib.2022.108774);
- leave one complete station out at a time;
- use only the existing train + validation period;
- compare nearest station, IDW p=1, and IDW p=2;
- keep the forecasting test period out of spatial comparison;
- treat reliability quantities as geometry proxies, not prediction intervals;
- do not infer minute-level support from the exact-time software interface.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_estimation import (
    STATION_BY_ID,
    estimate_deployment_pm25,
    estimate_oracle_pm25,
    generate_analysis_outputs,
)

In [ ]:
outputs = generate_analysis_outputs(
    PROJECT_ROOT / "data/processed/airquality_hcmc_clean.csv",
    PROJECT_ROOT / "reports",
)

geometry = outputs["geometry"]
distances = outputs["distances"]
metrics = outputs["metrics"]
temporal = outputs["temporal"]
reliability = outputs["reliability"]

geometry[["station_id", "latitude", "longitude", "station_type", "location"]]

In [ ]:
pooled = metrics.loc[metrics["held_out_station"].eq("ALL")]
held_out = metrics.loc[~metrics["held_out_station"].eq("ALL")]

print("Pooled LOSO development results")
display(pooled)
print("Held-out station results")
display(held_out)
print("Temporal robustness")
display(temporal)
print("Reliability proxies (not calibrated intervals)")
display(reliability)

In [ ]:
display(Image(filename=PROJECT_ROOT / "reports/figures/spatial_station_map.png"))
display(Image(filename=PROJECT_ROOT / "reports/figures/spatial_prediction_example.png"))

## Oracle versus deployment boundary

Both modes call the same spatial algorithm. Their only difference is the provenance of `station_values`:

- oracle values are actual observations at T and are used only to evaluate spatial interpolation;
- deployment values must be forecasts produced without observing T.

The spatial deployment wrapper has no dataset argument, so it cannot retrieve future observed PM2.5.

In [ ]:
location_x = (10.79, 106.67)

# Use one existing validation timestamp with six saved t+1h V1 forecasts.
# The deployment mapping contains the `prediction` field only, never target_pm25.
forecast_rows = pd.read_csv(
    PROJECT_ROOT / "data/processed/xgboost_forecasting_predictions.csv",
    parse_dates=["target_time"],
    low_memory=False,
)
forecast_rows = forecast_rows.loc[
    forecast_rows["split"].eq("validation")
    & forecast_rows["model"].eq("xgboost_v1")
    & forecast_rows["horizon_hours"].eq(1)
]
complete_times = forecast_rows.groupby("target_time")["Station_No"].nunique()
target_time = complete_times.loc[complete_times.eq(6)].index[0]
selected_forecasts = forecast_rows.loc[forecast_rows["target_time"].eq(target_time)]
forecasted_values = dict(
    zip(selected_forecasts["Station_No"], selected_forecasts["prediction"])
)

clean = pd.read_csv(
    PROJECT_ROOT / "data/processed/airquality_hcmc_clean.csv",
    parse_dates=["date"],
)
observed_rows = clean.loc[clean["date"].eq(target_time)].dropna(subset=["PM2.5"])
observed_values = dict(zip(observed_rows["Station_No"], observed_rows["PM2.5"]))

oracle_estimate = estimate_oracle_pm25(
    *location_x, target_time, observed_values, method="idw", power=2
)
deployment_estimate = estimate_deployment_pm25(
    *location_x, target_time, forecasted_values, method="idw", power=2
)

pd.DataFrame(
    {
        "mode": ["spatial oracle (observed station values)", "deployment (saved station forecasts)"],
        "target_time": [target_time, target_time],
        "predicted_pm25": [oracle_estimate, deployment_estimate],
        "station_value_count": [len(observed_values), len(forecasted_values)],
    }
)